# Add the five legal codes to the existing article dataset

This notebook preserves the current `data/dense_retrieval/articles.jsonl` records and appends the five structured legal-code sources from `data/legal_codes/source/articles_chapter.json` and `data/legal_codes/source/codes.json`. It does not add Labour Law because Labour Law is already present in the existing dataset.

`COMMIT = False` is intentional: first inspect the merged output and validation report. Set it to `True` only when the result is approved.

In [5]:
from pathlib import Path
from collections import Counter
import json, re, shutil, unicodedata

ROOT = Path.cwd()
if not (ROOT / 'apps').exists(): ROOT = Path('/Users/nak/CADT/projects/legal-rag')
CURRENT_ARTICLES = ROOT / 'data/dense_retrieval/articles.jsonl'
ARTICLES_SOURCE = ROOT / 'data/legal_codes/source/articles_chapter.json'
CODES_SOURCE = ROOT / 'data/legal_codes/source/codes.json'
OUTPUT_PREVIEW = ROOT / 'data/legal_codes/normalized/articles_with_core_codes.preview.jsonl'
OUTPUT_ACTIVE = ROOT / 'data/dense_retrieval/articles.jsonl'
COMMIT = True
EXPECTED_COUNTS = {
    'Constitution': 160,
    'Criminal Code': 672,
    'Criminal Procedure Code': 612,
    'Civil Code': 1305,
    'Civil Procedure Code': 588,
}
SLUGS = {
    'Constitution': 'constitution',
    'Criminal Code': 'criminal-code',
    'Criminal Procedure Code': 'criminal-procedure-code',
    'Civil Code': 'civil-code',
    'Civil Procedure Code': 'civil-procedure-code',
}
print('Current article file:', CURRENT_ARTICLES)
print('Commit mode:', COMMIT)

Current article file: /Users/nak/CADT/projects/legal-rag/data/dense_retrieval/articles.jsonl
Commit mode: True


In [6]:
def clean(value):
    if value is None: return ''
    return unicodedata.normalize('NFC', str(value).replace('\r', '').strip())

def article_number(record):
    match = re.search(r'មាត្រា\s*([0-9០-៩]+)', clean(record.get('name')))
    return int(match.group(1)) if match else record.get('id')

def code_record(source_name, record, code_meta, ordinal):
    slug = SLUGS[source_name]
    number = article_number(record)
    title = clean(code_meta.get('title') or record.get('book') or source_name)
    text = clean(record.get('description'))
    chapter = clean(record.get('chapter'))
    source_article_id = str(record.get('id'))
    return {
        'article_id': f'{slug}-article-{ordinal}',
        'document_id': slug,
        'article_number': number,
        'chapter_number': None,
        'chapter_title': chapter,
        'section_number': None,
        'section_title': '',
        'title': title,
        'document_type': 'Law Code',
        'domain': 'Khmer Legal Codes',
        'source_url': clean(code_meta.get('source')) or clean(code_meta.get('pdf_url')),
        'pdf_url': clean(code_meta.get('pdf_url')),
        'text_file': '',
        'preamble': '',
        'text': text,
        'search_text': clean(f'{title}\n{number}\n{chapter}\n{text}'),
        'normalization_version': 'structured-code-v1',
        'source_code_id': source_name,
        'source_article_id': source_article_id,
        'name_original': clean(record.get('name')),
        'name_en': clean(record.get('name_en')),
        'text_en': clean(record.get('description_en')),
    }

current = [json.loads(line) for line in CURRENT_ARTICLES.read_text(encoding='utf-8').splitlines() if line.strip()]
source_data = json.loads(ARTICLES_SOURCE.read_text(encoding='utf-8'))
code_metadata = {str(item['id']): item for item in json.loads(CODES_SOURCE.read_text(encoding='utf-8'))}
print('Existing records:', len(current))
print('Input source counts:', {name: len(source_data.get(name, [])) for name in EXPECTED_COUNTS})

Existing records: 3301
Input source counts: {'Constitution': 160, 'Criminal Code': 673, 'Criminal Procedure Code': 612, 'Civil Code': 1305, 'Civil Procedure Code': 588}


In [7]:
new_records = []
source_report = {}
for source_name, expected in EXPECTED_COUNTS.items():
    records = source_data[source_name]
    # The source contains two records labelled 437; the second is explicitly marked as a duplicate.
    if source_name == 'Criminal Code':
        records = [r for r in records if 'ស្ទួន' not in clean(r.get('name'))]
    if len(records) != expected:
        raise ValueError(f'{source_name}: expected {expected}, received {len(records)} after duplicate handling')
    metadata = code_metadata[source_name]
    converted = [code_record(source_name, record, metadata, ordinal) for ordinal, record in enumerate(records, 1)]
    if any(not item['text'] for item in converted):
        raise ValueError(f'{source_name}: at least one article has empty text')
    source_report[source_name] = len(converted)
    new_records.extend(converted)

existing_ids = {str(item['article_id']) for item in current}
new_ids = [str(item['article_id']) for item in new_records]
if len(new_ids) != len(set(new_ids)):
    raise ValueError('New source records contain duplicate article_id values')
id_collisions = sorted(existing_ids & set(new_ids))
if id_collisions:
    raise ValueError(f'Article ID collisions with existing dataset: {id_collisions[:10]}')

existing_texts = {clean(item.get('text')) for item in current}
exact_text_duplicates = [item['article_id'] for item in new_records if clean(item['text']) in existing_texts]
merged = current + [item for item in new_records if item['article_id'] not in exact_text_duplicates]
print('Source report:', source_report)
print('New records:', len(new_records))
print('Exact text duplicates skipped:', len(exact_text_duplicates))
print('Merged records:', len(merged))
if len(merged) != len(current) + len(new_records) - len(exact_text_duplicates): raise AssertionError('Merge count mismatch')

Source report: {'Constitution': 160, 'Criminal Code': 672, 'Criminal Procedure Code': 612, 'Civil Code': 1305, 'Civil Procedure Code': 588}
New records: 3337
Exact text duplicates skipped: 0
Merged records: 6638


In [8]:
target = OUTPUT_ACTIVE if COMMIT else OUTPUT_PREVIEW
target.parent.mkdir(parents=True, exist_ok=True)
if COMMIT:
    backup = OUTPUT_ACTIVE.with_suffix('.jsonl.pre-core-merge')
    if not backup.exists(): shutil.copy2(OUTPUT_ACTIVE, backup)
with target.open('w', encoding='utf-8') as stream:
    for item in merged:
        stream.write(json.dumps(item, ensure_ascii=False) + '\n')
print('Wrote:', target)
print('Commit mode:', COMMIT)
print('Next step after approval: rebuild chunks, embeddings, ChromaDB, and BM25 from the merged articles.jsonl.')

Wrote: /Users/nak/CADT/projects/legal-rag/data/dense_retrieval/articles.jsonl
Commit mode: True
Next step after approval: rebuild chunks, embeddings, ChromaDB, and BM25 from the merged articles.jsonl.


## Safety notes

- Labour Law is not imported from the five-code source because it already exists in the current dataset.
- The Criminal Code record marked `(ស្ទួន)` is excluded so the source contributes the requested 672 records.
- The default preview mode does not overwrite the active article file.
- After reviewing the counts and duplicate report, set `COMMIT = True` and rerun this notebook.
- Existing chunks, embeddings, ChromaDB, and BM25 must then be rebuilt because the article corpus has changed.